# v2 validation check — team-archetype grid → LR (polynomial), HOME market

A different **region definition** from the 4D-matchup notebooks. Instead of clustering the whole match on
four numbers at once, here we:

1. cluster individual **teams** in 2D (their own goals for / against) into a few archetypes,
2. label each match by the **pair** (home archetype, away archetype) — a grid of type-vs-type cells,
3. keep the grid cells where home wins happened more often than the market priced them (training fold only),
4. fit LR + polynomial and bet home where the model beats the implied probability by `buffer`.

Motivation: an earlier EDA snippet suggested matches between two teams of the same low-scoring archetype were
good for the home side. The flat 4D-matchup clustering could not isolate that pocket (it may merge it with
neighbours); the grid names every type-vs-type combination explicitly, so it can.

`running` vs `final` here refers only to which stats the **team archetypes** are fit on; matches are always
assigned, and the model fit, on running (point-in-time) stats. Rule #0: archetypes and cell selection are fit
on the training fold only.

In [1]:
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

CONFIG = {
    "n_team_types": 4,      # team archetypes -> n_team_types^2 grid cells (reference; swept below)
    "buffer": 0.02,
    "min_game_number": 8,
}

OUTCOME = "t_home_flg"                                             # betting the home win
IMPLIED = {"mrkt": "mrkt_home_impl", "b365": "b365_home_impl"}     # implied home-win prob = 1 / home odds

# The LR bet-signal model still uses the full 4D matchup (both teams' running for/against).
RUNNING_FEATURES = [
    "hmt_season_goals_for_avg", "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg", "awt_season_goals_agst_avg",
]
# The team clustering uses one team at a time in 2D. Running (point-in-time) and final (end-of-season) forms.
HOME_RUN, AWAY_RUN = ["hmt_season_goals_for_avg", "hmt_season_goals_agst_avg"], ["awt_season_goals_for_avg", "awt_season_goals_agst_avg"]
HOME_FIN, AWAY_FIN = ["home_for_final", "home_agst_final"], ["away_for_final", "away_agst_final"]
TEAM_MAPS = {"running": (HOME_RUN, AWAY_RUN), "final": (HOME_FIN, AWAY_FIN)}

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

final_stats = pd.read_parquet(f"C:/Users/twluc/frame/01_data/02_features/{GROUP}/team_season_final.parquet")
final_stats = final_stats[["team", "season", "season_goals_for_avg", "season_goals_agst_avg"]]
home_final = final_stats.rename(columns={"team": "hmt_name",
    "season_goals_for_avg": "home_for_final", "season_goals_agst_avg": "home_agst_final"})
away_final = final_stats.rename(columns={"team": "awt_name",
    "season_goals_for_avg": "away_for_final", "season_goals_agst_avg": "away_agst_final"})
abt = abt.merge(home_final, on=["hmt_name", "season"], how="left").merge(away_final, on=["awt_name", "season"], how="left")

matches = abt[
    (abt["hmt_game_number"] > CONFIG["min_game_number"])
    & (abt["awt_game_number"] > CONFIG["min_game_number"])
].dropna(subset=RUNNING_FEATURES + HOME_FIN + AWAY_FIN + list(IMPLIED.values()) + [OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [2]:
def pooled_roi(implied, won):
    staked = implied.sum()
    return (won.sum() - staked) / staked if staked > 0 else np.nan

def roi_grouped(bets, keys):
    totals = bets.groupby(keys).agg(won=("won", "sum"), staked=("implied", "sum"))
    return (totals["won"] - totals["staked"]) / totals["staked"]

def season_folds(data):
    for season in sorted(data["season"].unique()):
        yield ~(data["season"] == season), data["season"] == season, season

def expanding_folds(data, min_train_seasons=2):
    seasons = sorted(data["season"].unique())
    for split in range(min_train_seasons, len(seasons)):
        yield data["season"].isin(seasons[:split]), data["season"] == seasons[split], seasons[split]

def build_signal_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
                         LogisticRegression(max_iter=3000))

# Keep the grid cells where the outcome happened more often than the market priced it, on training data only.
def profitable_cells(train, cell_labels, min_matches=100):
    frame = pd.DataFrame({"cell": cell_labels, "outcome": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED["mrkt"]].to_numpy()})
    totals = frame.groupby("cell").agg(outcome=("outcome", "mean"), implied=("implied", "mean"), n=("outcome", "size"))
    return set(totals.index[(totals["n"] >= min_matches) & (totals["outcome"] > totals["implied"])])

In [3]:
# Honest per-fold run. `centroid_map` picks what the team archetypes are fit on ("running"/"final").
def collect_bets(folds, n_team_types, odds_source, centroid_map):
    home_map, away_map = TEAM_MAPS[centroid_map]
    implied_column = IMPLIED[odds_source]
    placed = []
    for train_mask, test_mask, fold_name in folds:
        train, test = matches[train_mask], matches[test_mask]

        # Team archetypes: cluster individual teams in 2D, both sides stacked, training seasons only.
        team_points = np.vstack([train[home_map].to_numpy(), train[away_map].to_numpy()])
        kmeans = KMeans(n_clusters=n_team_types, random_state=0, n_init=10).fit(team_points)

        # A match's cell = (home archetype, away archetype), always assigned by running stats.
        def cell_labels(frame):
            home_type = kmeans.predict(frame[HOME_RUN].to_numpy())
            away_type = kmeans.predict(frame[AWAY_RUN].to_numpy())
            return np.array([f"{h}-{a}" for h, a in zip(home_type, away_type)])

        train_cells = cell_labels(train)
        kept = profitable_cells(train, train_cells)
        test_cells = cell_labels(test)
        in_region_test = np.isin(test_cells, list(kept))
        if not in_region_test.any():
            continue
        in_region_train = np.isin(train_cells, list(kept))

        model = build_signal_model()
        model.fit(train[in_region_train][RUNNING_FEATURES], train[in_region_train][OUTCOME].astype(int))

        region = test[in_region_test].copy()
        region["model_prob"] = model.predict_proba(region[RUNNING_FEATURES])[:, list(model.classes_).index(1)]
        region["implied"] = region[implied_column]
        region["won"] = region[OUTCOME].astype(int)
        region["fold"] = fold_name

        is_bet = region["model_prob"] > region["implied"] + CONFIG["buffer"]
        placed.append(region.loc[is_bet, ["implied", "won", "league", "season", "fold"]])

    if not placed:
        return pd.DataFrame(columns=["implied", "won", "league", "season", "fold"])
    return pd.concat(placed, ignore_index=True)

## Gate 1 — time robustness (leave-one-season-out)

In [4]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"[{centroid_map:7s}] no bets placed under this pipeline"); continue
    per_season = roi_grouped(bets, "fold")
    positive = int((per_season > 0).sum())
    print(f"[{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          f"seasons +{positive}/{len(per_season)} {'PASS' if positive >= 3 else 'FAIL'}   ({len(bets)} bets)   "
          + "  ".join(f"{season}:{value:+.3f}" for season, value in per_season.items()))

[running] pooled +0.0309   seasons +3/4 PASS   (551 bets)   2223:+0.179  2324:+0.001  2425:-0.073  2526:+0.078


[final  ] pooled -0.0162   seasons +1/4 FAIL   (741 bets)   2223:+0.115  2324:-0.082  2425:-0.080  2526:-0.004


## Gate 2 — league robustness (survive removing the best league)

In [5]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"[{centroid_map:7s}] no bets placed under this pipeline"); continue
    without_league = {league: pooled_roi(bets[bets["league"] != league]["implied"], bets[bets["league"] != league]["won"])
                      for league in bets["league"].unique()}
    best_league = min(without_league, key=without_league.get)
    per_league = roi_grouped(bets, "league").sort_values()
    print(f"[{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          f"best league = {best_league} -> without it {without_league[best_league]:+.4f}   "
          f"GATE 2 {'PASS' if without_league[best_league] > 0 else 'FAIL'}")
    print("          per league: " + "   ".join(f"{league}:{value:+.3f}" for league, value in per_league.items()))

[running] pooled +0.0309   best league = spain -> without it +0.0039   GATE 2 PASS
          per league: italy:-0.064   germany:-0.055   france:+0.068   england:+0.070   spain:+0.195


[final  ] pooled -0.0162   best league = england -> without it -0.0460   GATE 2 FAIL
          per league: italy:-0.080   spain:-0.068   germany:-0.026   france:-0.008   england:+0.070


## GATE 3 — walk-forward floor, and knob robustness (sweep number of team archetypes)
Walk-forward trains only on earlier seasons. **GATE 3 fails** if any test season has ROI below **-0.05**, or if
**both** test seasons are negative. Sweeping the number of team archetypes stays a diagnostic.

In [6]:
_GREEN, _RED, _RESET = chr(27) + "[92m", chr(27) + "[91m", chr(27) + "[0m"
print("walk-forward (b365)  -  GATE 3 fails if any test season < -0.05 ROI, or both test seasons are negative:")
for centroid_map in ["running", "final"]:
    bets = collect_bets(expanding_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"  [{centroid_map:7s}] no bets"); continue
    per_fold = roi_grouped(bets, "fold")
    gate3_fail = bool((per_fold < -0.05).any()) or bool((per_fold < 0).all())
    verdict = f"{_RED}GATE 3 FAIL{_RESET}" if gate3_fail else f"{_GREEN}GATE 3 PASS{_RESET}"
    print(f"  [{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          + "   ".join(f"{fold}:{value:+.3f}" for fold, value in per_fold.items())
          + f"   {verdict}")

print("\nknob robustness - pooled ROI (b365) by number of team archetypes (cells = types^2):")
print("  types  cells  running   final")
for n_types in [2, 3, 4, 5]:
    run_bets = collect_bets(season_folds(matches), n_types, "b365", "running")
    fin_bets = collect_bets(season_folds(matches), n_types, "b365", "final")
    print(f"    {n_types}     {n_types**2:2d}   {pooled_roi(run_bets['implied'], run_bets['won']):+.4f}  "
          f"{pooled_roi(fin_bets['implied'], fin_bets['won']):+.4f}")

walk-forward (b365)  -  GATE 3 fails if any test season < -0.05 ROI, or both test seasons are negative:
  [running] pooled -0.0091   2425:-0.067   2526:+0.078   GATE 3 FAIL


  [final  ] pooled -0.0471   2425:-0.071   2526:-0.004   GATE 3 FAIL

knob robustness - pooled ROI (b365) by number of team archetypes (cells = types^2):
  types  cells  running   final


    2      4   -0.0921  -0.1432


    3      9   -0.0814  -0.0762


    4     16   +0.0309  -0.0162


    5     25   -0.0375  -0.0356


## Descriptive (in-sample, not a gate) — ROI per grid cell
Which type-vs-type cell drives the home market? Team archetypes fit on all matches (running); each cell is
annotated with the two archetypes' average (goals for, goals against). In-sample, so treat as a map of where
to look, not evidence.

In [7]:
team_points = np.vstack([matches[HOME_RUN].to_numpy(), matches[AWAY_RUN].to_numpy()])
km = KMeans(n_clusters=CONFIG["n_team_types"], random_state=0, n_init=10).fit(team_points)
print("team archetypes (for, agst):")
for i, centre in enumerate(km.cluster_centers_):
    print(f"  type {i}: for={centre[0]:.2f}  agst={centre[1]:.2f}")

home_type = km.predict(matches[HOME_RUN].to_numpy())
away_type = km.predict(matches[AWAY_RUN].to_numpy())
grid = pd.DataFrame({"cell": [f"{h}-{a}" for h, a in zip(home_type, away_type)],
                     "won": matches[OUTCOME].astype(int).to_numpy(), "implied": matches["b365_home_impl"].to_numpy()})
per_cell = grid.groupby("cell").agg(won=("won", "sum"), staked=("implied", "sum"), n=("won", "size"))
per_cell["roi"] = (per_cell["won"] - per_cell["staked"]) / per_cell["staked"]
print("\nROI per cell (home-type - away-type), in-sample b365:")
print(per_cell[["n", "roi"]].sort_values("roi", ascending=False).round(3).to_string())

team archetypes (for, agst):
  type 0: for=2.39  agst=0.95
  type 1: for=1.03  agst=1.29
  type 2: for=1.13  agst=1.92
  type 3: for=1.68  agst=1.24

ROI per cell (home-type - away-type), in-sample b365:
        n    roi
cell            
0-0    54  0.174
3-3   541  0.067
0-1   184  0.014
1-3   543 -0.023
0-2   164 -0.024
0-3   195 -0.028
1-2   381 -0.038
2-2   430 -0.044
1-1   696 -0.068
3-1   535 -0.077
3-0   190 -0.104
3-2   425 -0.109
2-0   164 -0.113
2-1   399 -0.152
1-0   187 -0.176
2-3   445 -0.206


## Diagnostic — where the edge lives (league × season), running map

In [8]:
bets = collect_bets(season_folds(matches), CONFIG["n_team_types"], "b365", "running")
if bets.empty:
    print("no bets placed under this pipeline")
else:
    breakdown = roi_grouped(bets, ["league", "season"]).unstack("season").round(3)
    print(f"ROI by league x season (b365), total {len(bets)} bets:")
    print(breakdown.to_string())

ROI by league x season (b365), total 551 bets:
season    2223   2324   2425   2526
league                             
england  0.270  0.012 -0.002  0.005
france   0.139 -0.224  0.046  0.194
germany  0.118  0.027 -0.553  0.109
italy    0.243 -0.022 -0.109 -0.271
spain    0.081  0.136  0.239  0.227


## Summary

In [9]:
# ---------------------------------------------------------------------------
# SUMMARY - the three gates at a glance (b365, running map, default k). GREEN = pass, RED = fail.
#   GATE 1  LOSO:               edge positive in at least 3 of 4 seasons.
#   GATE 2  drop-best-league:   pooled edge survives removing the single best league.
#   GATE 3  walk-forward floor: no test season below -0.05 ROI, and not both test seasons negative.
# Also writes/refreshes this notebook's one-line result in model_checks/results.txt.
# ---------------------------------------------------------------------------
NOTEBOOK = "goals_foragst_team_kmeans_lr_home"

_GREEN, _RED, _RESET = chr(27) + "[92m", chr(27) + "[91m", chr(27) + "[0m"
def _paint(ok):
    return f"{_GREEN}PASS{_RESET}" if ok else f"{_RED}FAIL{_RESET}"

k = CONFIG.get("n_clusters", CONFIG.get("n_team_types"))
bets = collect_bets(season_folds(matches), k, "b365", "running")
walk = collect_bets(expanding_folds(matches), k, "b365", "running")

if bets.empty:
    print("SUMMARY: no bets placed under this pipeline")
else:
    per_season = roi_grouped(bets, "fold")
    n_positive = int((per_season > 0).sum())
    gate1_pass = n_positive >= 3

    without_league = {league: pooled_roi(bets[bets["league"] != league]["implied"], bets[bets["league"] != league]["won"])
                      for league in bets["league"].unique()}
    gate2_pass = min(without_league.values()) > 0

    wf = roi_grouped(walk, "fold")
    gate3_pass = (not walk.empty) and bool((wf >= -0.05).all()) and not bool((wf < 0).all())

    pooled = pooled_roi(bets["implied"], bets["won"])
    spread = roi_grouped(bets, ["league", "season"]).std()  # how noisy the per-cell breakdown is
    wf_detail = "  ".join(f"{fold}:{value:+.3f}" for fold, value in wf.items()) if not walk.empty else "no bets"

    print(f"SUMMARY (b365, running map, k={k}, group={GROUP})")
    print(f"  pooled ROI                         {pooled:+.4f}  ({len(bets)} bets)")
    print(f"  GATE 1  (LOSO >= 3/4 seasons)       {_paint(gate1_pass)}  (+{n_positive}/{len(per_season)})")
    print(f"  GATE 2  (drop best league)          {_paint(gate2_pass)}  (worst {min(without_league.values()):+.4f})")
    print(f"  GATE 3  (walk-forward floor)        {_paint(gate3_pass)}  ({wf_detail})")
    print(f"  diagnostic spread (std league x season ROI)   {spread:.3f}")

    from evaluation.results_registry import upsert
    _verdict = lambda ok: "PASS" if ok else "FAIL"
    upsert(source=NOTEBOOK, space="goals_foragst", target="home",
           abt_filter="major, gn>8, dropna",
           segmentation="KMeans k=4 team archetypes → 4x4 grid", segment_filter="train-profitable cells",
           g1=_verdict(gate1_pass), g2=_verdict(gate2_pass), g3=_verdict(gate3_pass),
           result=f"pooled {pooled:+.3f} ({len(bets)} bets)", roi_std=f"{spread:.3f}")
    print(f"\nlogged to model_checks/results.md: {NOTEBOOK}")


SUMMARY (b365, running map, k=4, group=major)
  pooled ROI                         +0.0309  (551 bets)
  GATE 1  (LOSO >= 3/4 seasons)       PASS  (+3/4)
  GATE 2  (drop best league)          PASS  (worst +0.0039)
  GATE 3  (walk-forward floor)        FAIL  (2425:-0.067  2526:+0.078)
  diagnostic spread (std league x season ROI)   0.201

logged to model_checks/results.md: goals_foragst_team_kmeans_lr_home
